In [17]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm_notebook
pd.set_option("display.max_columns", 500)

EVENTS_DIR = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_DIR = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"
VAULTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/common/vaults_list.csv"
SPIKES_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/all_spikes_dataset.csv"
SUPPLIERS_SHARE_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_suppliers_share.csv"
BORROWERS_SHARE_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_borrowers_share.csv"

INTERESTING_MARKETS = [
    "eth_syrupusdc_usdc",
    "eth_usr_usdc",
    "eth_rlp_usdc",
    "eth_wstusr_usdc",
    "eth_fxsave_usdc",
    "eth_siusd_usdc",
    "eth_reusd_usdc",
    "eth_sdeusd_usdc",
    "eth_slvlusd_usdc",
    "eth_stcusd_usdc",
    "eth_csusdl_usdc",
    "eth_mhyper_usdc",
    "eth_mapollo_usdc",
    "eth_mF-ONE_usdc",
    "eth_wsrusd_usdc",
]

def load_market_events(market_name):
    file_path = os.path.join(EVENTS_DIR, f"{market_name}.csv")
    df = pd.read_csv(file_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df

def load_market_hourly(market_name):
    file_path = os.path.join(HOURLY_DIR, f"{market_name}.csv")
    df = pd.read_csv(file_path)
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df

def load_spikes():
    df = pd.read_csv(SPIKES_PATH)
    df['spike_trigger_datetime'] = pd.to_datetime(df['spike_trigger_datetime'])
    df['spike_recovery_datetime'] = pd.to_datetime(df['spike_recovery_datetime'])
    return df

def load_suppliers_share():
    df = pd.read_csv(SUPPLIERS_SHARE_PATH)
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df

def load_borrowers_share():
    df = pd.read_csv(BORROWERS_SHARE_PATH)
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df

def get_closest_value(df, timestamp, column):
    idx = df['timestamp'].searchsorted(timestamp, side='right') - 1
    if idx < 0:
        return np.nan
    return df.iloc[idx][column]

def get_share_at_timestamp(share_df, market, timestamp, side):
    sub = share_df[(share_df['market'] == market) & (share_df['side'] == side)]
    if sub.empty:
        return None
    idx = sub['timestamp'].searchsorted(timestamp, side='right') - 1
    if idx < 0:
        return None
    return sub.iloc[idx]

def extract_user_positions(market_events):
    market_events = market_events.sort_values(['user_address', 'timestamp']).copy()
    positions = []
    
    for user, user_df in market_events.groupby('user_address'):
        user_df = user_df.sort_values('timestamp').reset_index(drop=True)
        open_indices = user_df[user_df['event_sequence_type'] == 'position_open'].index.tolist()
        close_indices = user_df[user_df['event_sequence_type'] == 'position_close'].index.tolist()
        
        for open_idx in open_indices:
            open_row = user_df.loc[open_idx]
            open_ts = open_row['timestamp']
            
            close_candidates = [c for c in close_indices if c > open_idx]
            if close_candidates:
                close_idx = close_candidates[0]
                close_row = user_df.loc[close_idx]
                close_ts = close_row['timestamp']
                position_events = user_df.loc[open_idx:close_idx]
                is_closed = True
            else:
                close_row = None
                close_ts = None
                position_events = user_df.loc[open_idx:]
                is_closed = False
            
            open_debt = open_row['debt_after']
            open_ltv = open_row['ltv_after']
            open_borrow_rate = open_row['borrow_rate_after']
            
            if is_closed:
                close_debt = close_row['debt_before']
                close_ltv = close_row['ltv_before']
                close_borrow_rate = close_row['borrow_rate_before']
            else:
                close_debt = None
                close_ltv = None
                close_borrow_rate = None
            
            debt_values = position_events['debt_after'].values
            ltv_values = position_events['ltv_after'].values
            max_debt = np.max(debt_values) if len(debt_values) > 0 else open_debt
            max_ltv = np.max(ltv_values) if len(ltv_values) > 0 else open_ltv
            
            position_events_ex_open = position_events.iloc[1:] if len(position_events) > 1 else pd.DataFrame()
            n_repays = (position_events_ex_open['type'] == 'MarketRepay').sum() if not position_events_ex_open.empty else 0
            n_borrows = (position_events_ex_open['type'] == 'MarketBorrow').sum() if not position_events_ex_open.empty else 0
            
            twenty_four_hours = 24 * 3600
            borrow_mask = (user_df['type'] == 'MarketBorrow') & \
                          (user_df['timestamp'] > open_ts) & \
                          (user_df['timestamp'] <= open_ts + twenty_four_hours)
            leverage_factor = borrow_mask.sum()
            
            positions.append({
                'user_address': user,
                'open_timestamp': open_ts,
                'open_datetime': open_row['datetime'],
                'open_debt': open_debt,
                'open_ltv': open_ltv,
                'open_borrow_rate': open_borrow_rate,
                'leverage_factor': leverage_factor,
                'close_timestamp': close_ts,
                'close_datetime': close_row['datetime'] if is_closed else None,
                'close_debt': close_debt,
                'close_ltv': close_ltv,
                'close_borrow_rate': close_borrow_rate,
                'max_debt': max_debt,
                'max_ltv': max_ltv,
                'n_repays': n_repays,
                'n_borrows': n_borrows,
                'is_closed': is_closed,
                'position_events': position_events,
                'user_events_df': user_df,
                'open_idx': open_idx,
                'close_idx': close_idx if is_closed else None
            })
    
    return positions

def compute_position_features(positions, market_events, market_hourly, market_spikes, suppliers_share, borrowers_share, market_name):
    if not positions:
        return pd.DataFrame()
    
    positions_df = pd.DataFrame([{
        'user_address': p['user_address'],
        'open_timestamp': p['open_timestamp'],
        'open_datetime': p['open_datetime'],
        'open_debt': p['open_debt'],
        'open_ltv': p['open_ltv'],
        'open_borrow_rate': p['open_borrow_rate'],
        'leverage_factor': p['leverage_factor'],
        'close_timestamp': p['close_timestamp'],
        'close_datetime': p['close_datetime'],
        'close_debt': p['close_debt'],
        'close_ltv': p['close_ltv'],
        'close_borrow_rate': p['close_borrow_rate'],
        'max_debt': p['max_debt'],
        'max_ltv': p['max_ltv'],
        'n_repays': p['n_repays'],
        'n_borrows': p['n_borrows'],
        'is_closed': p['is_closed'],
        'position_events': p['position_events'],
        'user_events_df': p['user_events_df'],
    } for p in positions])
    
    hourly_sorted = market_hourly.sort_values('timestamp')
    hourly_ts = hourly_sorted['timestamp'].values
    hourly_total_borrow = hourly_sorted['total_borrow'].values
    hourly_total_supply = hourly_sorted['total_supply'].values
    hourly_utilization = hourly_sorted['utilization'].values
    hourly_borrow_rate = hourly_sorted['borrow_rate'].values
    
    def get_hourly_at(ts):
        idx = np.searchsorted(hourly_ts, ts, side='right') - 1
        if idx < 0:
            return None, None, None, None
        return hourly_total_borrow[idx], hourly_total_supply[idx], hourly_utilization[idx], hourly_borrow_rate[idx]
    
    open_vals = [get_hourly_at(ts) for ts in positions_df['open_timestamp']]
    positions_df['total_borrow_open'] = [v[0] for v in open_vals]
    positions_df['total_supply_open'] = [v[1] for v in open_vals]
    positions_df['utilization_open'] = [v[2] for v in open_vals]
    positions_df['total_debt_open'] = positions_df['total_borrow_open']
    positions_df['total_liquidity_open'] = positions_df['total_supply_open'] - positions_df['total_borrow_open']
    positions_df['position_size_share_open'] = positions_df['open_debt'] / positions_df['total_borrow_open'].replace(0, np.nan)
    
    max_debt_ts_list = []
    for _, row in positions_df.iterrows():
        pos_events = row['position_events']
        if len(pos_events) > 0:
            max_debt_ts = pos_events[pos_events['debt_after'] == row['max_debt']]['timestamp'].iloc[0]
        else:
            max_debt_ts = row['open_timestamp']
        max_debt_ts_list.append(max_debt_ts)
    positions_df['max_debt_ts'] = max_debt_ts_list
    
    max_borrow_vals = [get_hourly_at(ts)[0] for ts in max_debt_ts_list]
    positions_df['total_borrow_max'] = max_borrow_vals
    positions_df['position_size_share_max'] = positions_df['max_debt'] / positions_df['total_borrow_max'].replace(0, np.nan)
    
    duration_hours = []
    time_to_first_action = []
    avg_time_between_actions = []
    max_time_between_actions = []
    n_actions_total = []
    avg_repay_ratio = []
    position_end_list = []
    
    for _, row in positions_df.iterrows():
        open_ts = row['open_timestamp']
        close_ts = row['close_timestamp'] if row['is_closed'] else None
        pos_events = row['position_events']
        
        if close_ts is None:
            last_event_ts = pos_events['timestamp'].max()
            duration_sec = last_event_ts - open_ts
            position_end = last_event_ts
        else:
            duration_sec = close_ts - open_ts
            position_end = close_ts
        duration_hours.append(duration_sec / 3600.0)
        position_end_list.append(position_end)
        
        actions = pos_events.iloc[1:] if len(pos_events) > 1 else pd.DataFrame()
        if not actions.empty:
            actions = actions[actions['type'].isin(['MarketBorrow', 'MarketRepay'])]
        n_act = len(actions)
        n_actions_total.append(n_act)
        
        if n_act > 0:
            act_ts = actions['timestamp'].values
            tfa = (act_ts[0] - open_ts) / 3600.0
            time_to_first_action.append(tfa)
            if n_act > 1:
                gaps = np.diff(act_ts) / 3600.0
                avg_time_between_actions.append(gaps.mean())
                max_time_between_actions.append(gaps.max())
            else:
                avg_time_between_actions.append(np.nan)
                max_time_between_actions.append(np.nan)
        else:
            time_to_first_action.append(np.nan)
            avg_time_between_actions.append(np.nan)
            max_time_between_actions.append(np.nan)
        
        repay_actions = pos_events[pos_events['type'] == 'MarketRepay']
        ratios = []
        for _, ra in repay_actions.iterrows():
            if ra['debt_before'] > 0:
                ratios.append(ra['assets_usd'] / ra['debt_before'])
        avg_repay_ratio.append(np.mean(ratios) if ratios else np.nan)
    
    positions_df['duration_hours'] = duration_hours
    positions_df['time_to_first_action'] = time_to_first_action
    positions_df['avg_time_between_actions'] = avg_time_between_actions
    positions_df['max_time_between_actions'] = max_time_between_actions
    positions_df['n_actions_total'] = n_actions_total
    positions_df['avg_repay_ratio'] = avg_repay_ratio
    positions_df['position_end'] = position_end_list
    
    if not market_spikes.empty:
        spike_starts = market_spikes['spike_trigger_timestamp'].values
        spike_ends = market_spikes['spike_recovery_timestamp'].values
        open_ts_arr = positions_df['open_timestamp'].values
        close_ts_arr = positions_df['close_timestamp'].values
        is_closed_arr = positions_df['is_closed'].values
        pos_end_arr = positions_df['position_end'].values
        
        was_active = np.zeros(len(positions_df), dtype=bool)
        num_spikes = np.zeros(len(positions_df), dtype=int)
        closed_during = np.zeros(len(positions_df), dtype=bool)
        
        for i in range(len(spike_starts)):
            s_start = spike_starts[i]
            s_end = spike_ends[i]
            overlap = (open_ts_arr <= s_end) & (pos_end_arr >= s_start)
            was_active |= overlap
            num_spikes += overlap
            
            if is_closed_arr.any():
                closed_mask = overlap & (close_ts_arr >= s_start) & (close_ts_arr <= s_end)
                closed_during |= closed_mask
        
        positions_df['was_active_during_spike'] = was_active
        positions_df['num_spikes_experienced'] = num_spikes
        positions_df['closed_during_spike'] = closed_during
    else:
        positions_df['was_active_during_spike'] = False
        positions_df['num_spikes_experienced'] = 0
        positions_df['closed_during_spike'] = False
    
    borrowers_market = borrowers_share[borrowers_share['market'] == market_name].sort_values('timestamp')
    borrowers_ts = borrowers_market['timestamp'].values
    borrowers_users = borrowers_market['user_address'].values
    borrowers_share_val = borrowers_market['share'].values
    
    def is_top_borrower(user, open_ts, close_ts, is_closed):
        idx_start = np.searchsorted(borrowers_ts, open_ts, side='left')
        if is_closed and close_ts is not None:
            idx_end = np.searchsorted(borrowers_ts, close_ts, side='right')
        else:
            idx_end = len(borrowers_ts)
        for i in range(idx_start, idx_end):
            if borrowers_users[i] == user and borrowers_share_val[i] > 0:
                return True
        return False
    
    debtors_rank_flags = []
    for _, row in positions_df.iterrows():
        debtors_rank_flags.append(
            is_top_borrower(row['user_address'], row['open_timestamp'], row['close_timestamp'], row['is_closed'])
        )
    positions_df['debtors_rank'] = debtors_rank_flags
    
    suppliers_market = suppliers_share[suppliers_share['market'] == market_name].sort_values('timestamp')
    suppliers_ts = suppliers_market['timestamp'].values
    suppliers_hhi = suppliers_market['hhi'].values
    
    def get_hhi_at(ts):
        idx = np.searchsorted(suppliers_ts, ts, side='right') - 1
        if idx < 0:
            return np.nan
        return suppliers_hhi[idx]
    
    positions_df['concentration_hhi_open'] = positions_df['open_timestamp'].apply(get_hhi_at)
    
    borrowers_market_open = borrowers_share[
        (borrowers_share['market'] == market_name) & 
        (borrowers_share['side'] == 'borrow') &
        (borrowers_share['user_address'] != 'other')
    ].sort_values(['timestamp', 'share'], ascending=[True, False])
    
    top3_map = {}
    for ts, group in borrowers_market_open.groupby('timestamp'):
        top3_map[ts] = group['share'].iloc[:3].sum()
    borrowers_open_ts_all = np.array(list(top3_map.keys()))
    borrowers_open_top3 = np.array(list(top3_map.values()))
    
    def get_top3_at(ts):
        idx = np.searchsorted(borrowers_open_ts_all, ts, side='right') - 1
        if idx < 0:
            return np.nan
        return borrowers_open_top3[idx]
    
    positions_df['top3_share_open'] = positions_df['open_timestamp'].apply(get_top3_at)
    
    avg_borrow_rates = []
    for _, row in positions_df.iterrows():
        mask = (hourly_ts >= row['open_timestamp']) & (hourly_ts <= row['position_end'])
        if mask.any():
            avg_borrow_rates.append(hourly_borrow_rate[mask].mean())
        else:
            avg_borrow_rates.append(row['open_borrow_rate'])
    positions_df['avg_borrow_rate_position'] = avg_borrow_rates
    
    result_cols = [
        'user_address', 'market', 'open_timestamp', 'open_datetime', 'open_debt', 'open_ltv',
        'open_borrow_rate', 'leverage_factor', 'close_timestamp', 'close_datetime', 'close_debt',
        'close_ltv', 'close_borrow_rate', 'max_debt', 'max_ltv', 'n_repays', 'n_borrows',
        'is_closed', 'duration_hours', 'time_to_first_action', 'avg_time_between_actions',
        'max_time_between_actions', 'n_actions_total', 'avg_repay_ratio', 'was_active_during_spike',
        'num_spikes_experienced', 'closed_during_spike', 'position_size_share_open',
        'position_size_share_max', 'debtors_rank', 'utilization_open', 'total_debt_open',
        'total_liquidity_open', 'concentration_hhi_open', 'top3_share_open',
        'avg_borrow_rate_position'
    ]
    
    positions_df['market'] = market_name
    result_df = positions_df[result_cols].sort_values('open_timestamp').reset_index(drop=True)
    result_df['position_index'] = result_df.index
    return result_df


def build_market_positions_dataset(market_name):
    events = load_market_events(market_name)
    hourly = load_market_hourly(market_name)
    spikes_all = load_spikes()
    spikes_all['spike_trigger_timestamp'] = spikes_all['spike_trigger_datetime'].astype(int) // 10**9
    spikes_all['spike_recovery_timestamp'] = spikes_all['spike_recovery_datetime'].astype(int) // 10**9
    market_spikes = spikes_all[spikes_all['market_name'] == market_name]
    suppliers_share = load_suppliers_share()
    borrowers_share = load_borrowers_share()
    
    positions_raw = extract_user_positions(events)
    positions_df = compute_position_features(
        positions_raw, events, hourly, market_spikes,
        suppliers_share, borrowers_share, market_name
    )
    return positions_df

def build_all_positions_dataset(market_list):
    all_positions = []
    for market in tqdm_notebook(market_list):
        try:
            market_positions = build_market_positions_dataset(market)
            parts = market.split('_')
            collateral_asset = parts[1] if len(parts) > 1 else None
            loan_asset = parts[2] if len(parts) > 2 else None
            market_positions["collateral_asset"] = collateral_asset
            market_positions["loan_asset"] = loan_asset

            all_positions.append(market_positions)
        except FileNotFoundError as e:
            print(f"Data files for market {market} not found, skipping. Error: {e}")
            continue
    if all_positions:
        final_df = pd.concat(all_positions, ignore_index=True)
        final_df = final_df.sort_values(['market', 'open_timestamp']).reset_index(drop=True)
        return final_df
    else:
        return pd.DataFrame()


In [20]:
dataset = build_all_positions_dataset(INTERESTING_MARKETS)
dataset.head(3)
output_path = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/users_positions/yb_tokens_positions.csv"
dataset.to_csv(output_path, index=False)
print(f"Dataset saved to {output_path}")

/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_81751/856417801.py:408: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for market in tqdm_notebook(market_list):


  0%|          | 0/15 [00:00<?, ?it/s]

/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_81751/856417801.py:34: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)
/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_81751/856417801.py:34: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Dataset saved to /Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/users_positions/yb_tokens_positions.csv


In [23]:
dataset.head(40)
# print(dataset.shape)
# dataset.isna().sum()

,user_address,market,open_timestamp,open_datetime,open_debt,open_ltv,open_borrow_rate,leverage_factor,close_timestamp,close_datetime,close_debt,close_ltv,close_borrow_rate,max_debt,max_ltv,n_repays,n_borrows,is_closed,duration_hours,time_to_first_action,avg_time_between_actions,max_time_between_actions,n_actions_total,avg_repay_ratio,was_active_during_spike,num_spikes_experienced,closed_during_spike,position_size_share_open,position_size_share_max,debtors_rank,utilization_open,total_debt_open,total_liquidity_open,concentration_hhi_open,top3_share_open,avg_borrow_rate_position,position_index,collateral_asset,loan_asset
0,0x074f6AE33344644a4eB3A455fc7F0930BdDf3fe2,eth_csusdl_usdc,1738592699,2025-02-03 14:24:59,99.996100,0.0,0.217090,8,1.740862e+09,2025-03-01 20:41:23,1.015515e+05,0.000000,0.053325,4.030601e+05,0.000000,2,10,True,630.273333,0.000000,57.293333,517.660000,12,0.463204,True,7,False,97.486339,0.372880,False,0.816103,1.025745e+00,2.311371e-01,NaN,NaN,0.119449,0,csusdl,usdc
1,0x074f6AE33344644a4eB3A455fc7F0930BdDf3fe2,eth_csusdl_usdc,1738592699,2025-02-03 14:24:59,99.996100,0.0,0.217090,8,1.740862e+09,2025-03-01 20:41:23,1.015515e+05,0.000000,0.053325,4.030601e+05,0.000000,2,9,True,630.273333,0.066667,63.016000,517.660000,11,0.463204,True,7,False,97.486339,0.372880,False,0.816103,1.025745e+00,2.311371e-01,NaN,NaN,0.119449,1,csusdl,usdc
2,0x6320203e9b9F78bb65ffAFF4c1CCEE88f0A872Ed,eth_csusdl_usdc,1738613087,2025-02-03 20:04:47,169984.870000,0.0,0.187312,3,1.740864e+09,2025-03-01 21:20:35,2.468048e+04,0.000000,0.052783,5.329995e+05,0.000000,5,3,True,625.263333,0.116667,89.306667,622.233333,8,0.582450,True,7,False,0.658560,0.724042,False,0.615652,2.581162e+05,1.611401e+05,NaN,NaN,0.118649,2,csusdl,usdc
3,0x6320203e9b9F78bb65ffAFF4c1CCEE88f0A872Ed,eth_csusdl_usdc,1738613087,2025-02-03 20:04:47,169984.870000,0.0,0.187312,3,1.740864e+09,2025-03-01 21:20:35,2.468048e+04,0.000000,0.052783,5.329995e+05,0.000000,5,4,True,625.263333,0.000000,78.157917,622.233333,9,0.582450,True,7,False,0.658560,0.724042,False,0.615652,2.581162e+05,1.611401e+05,NaN,NaN,0.118649,3,csusdl,usdc
4,0x9dEfF269B22849889cAE9A965769f576a1e72d27,eth_csusdl_usdc,1738844411,2025-02-06 12:20:11,69776.667485,0.0,0.183497,1,1.744818e+09,2025-04-16 15:35:59,4.444467e+04,0.879726,0.054577,1.447712e+05,0.878925,2,4,True,1659.263333,21.323333,327.588000,1500.710000,6,0.867974,True,71,False,0.084352,0.150772,False,0.827203,8.272036e+05,1.727970e+05,NaN,NaN,0.075997,4,csusdl,usdc
5,0x9dEfF269B22849889cAE9A965769f576a1e72d27,eth_csusdl_usdc,1738844411,2025-02-06 12:20:11,69776.667485,0.0,0.183497,1,1.744818e+09,2025-04-16 15:35:59,4.444467e+04,0.879726,0.054577,1.447712e+05,0.878925,2,4,True,1659.263333,21.323333,327.588000,1500.710000,6,0.867974,True,71,False,0.084352,0.150772,False,0.827203,8.272036e+05,1.727970e+05,NaN,NaN,0.075997,5,csusdl,usdc
6,0x977767f401Fc909614905857c9174bd08af02363,eth_csusdl_usdc,1739177339,2025-02-10 08:48:59,8499.991500,0.0,0.142668,2,1.752809e+09,2025-07-18 03:19:35,5.659519e+02,0.429925,0.108729,2.186598e+04,0.883453,4,3,True,3786.510000,0.000000,631.085000,3786.260000,7,0.940386,True,181,False,0.009432,0.024263,False,0.686063,9.012154e+05,4.123892e+05,NaN,NaN,0.067150,6,csusdl,usdc
7,0x977767f401Fc909614905857c9174bd08af02363,eth_csusdl_usdc,1739177339,2025-02-10 08:48:59,8499.991500,0.0,0.142668,2,1.752809e+09,2025-07-18 03:19:35,5.659519e+02,0.429925,0.108729,2.186598e+04,0.883453,4,2,True,3786.510000,0.053333,757.291333,3786.260000,6,0.940386,True,181,False,0.009432,0.024263,False,0.686063,9.012154e+05,4.123892e+05,NaN,NaN,0.067150,7,csusdl,usdc
8,0x6b019b4FD5D79b79A5Cb93e6E8b9bE0E7bc8950F,eth_csusdl_usdc,1739225087,2025-02-10 22:04:47,999.995000,0.0,0.142482,0,NaN,NaT,NaN,NaN,NaN,1.009821e+03,0.903354,0,1,False,7554.733333,5033.730000,NaN,NaN,1,NaN,True,247,False,0.001077,0.001534,False,0.722129,9.284453e+05,3.572605e+05,NaN,NaN,0.144833,8,csusdl,usdc
9,0x6b019b4FD5D79b79A5Cb93e6E8b9bE0E7bc8950F,eth_csusdl_usdc,1739225087,2025-02-10